## SHAP Value Calculation for Time Series (BiLSTM)

In this notebook, SHAP (SHapley Additive exPlanations) values are computed **manually** using two custom approaches, since the official `shap` library does not integrate well with BiLSTM-based time series models.

---

### 📌 Theoretical Definition

The Shapley value of a feature *i* is defined as:

$$
\phi_i(f) = \sum_{S \subseteq N \setminus \{i\}}
\frac{|S|! \, (|N|-|S|-1)!}{|N|!}
\Big[ f(S \cup \{i\}) - f(S) \Big]
$$

Where:  
- \(N\): the full feature set  
- \(S\): a subset of features not containing \(i\)  
- \(f(S)\): model prediction using only features in \(S\)  
- \(\phi_i\): the average marginal contribution of feature \(i\).  

This is the exact game-theoretic formulation, but it is computationally intractable for high-dimensional inputs (exponential in the number of features).

---

### ⚡ Practical Implementations in This Notebook

1. **Masking-based SHAP**  
   - Specific features or timesteps are replaced with baseline values (from a background dataset).  
   - Predictions with and without the feature are compared:  

   $$
   \Delta_i = f(S \cup \{i\}) - f(S)
   $$

   - This directly measures the marginal contribution of each feature/timestep, averaged across baselines.

2. **Sampling-based SHAP (Monte Carlo Approximation)**  
   - Random coalitions \(S\) of features are sampled.  
   - For each coalition, the marginal contribution of adding feature \(i\) is computed:  

   $$
   \phi_i(f) \approx \frac{1}{M} \sum_{m=1}^{M} 
   \Big[ f(S_m \cup \{i\}) - f(S_m) \Big]
   $$

   - Repeated many times (\(M\)) to approximate the true Shapley value.  

---

### ✅ Why This Approach?

- Ensures **model-agnostic correctness** (faithful to the original Shapley definition).  
- Works with **BiLSTM time series models**, where existing SHAP explainers are not directly compatible.  
- Produces results consistent with Shapley properties (e.g., attributions summing close to zero).  

The trade-off is **higher computational cost**, but GPU acceleration makes the approach feasible for real-world time series datasets.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Layer
import joblib
import os
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# GPU Konfigürasyonu
def configure_gpu():
    """GPU ayarlarını yapılandır"""
    print("🔧 GPU Konfigürasyonu başlatılıyor...")
    
    gpus = tf.config.experimental.list_physical_devices('GPU')
    print(f"🎯 Bulunan GPU sayısı: {len(gpus)}")
    
    if gpus:
        try:
            logical_gpus = tf.config.experimental.list_logical_devices('GPU')
            if len(logical_gpus) > len(gpus):
                print("⚠️ Virtual GPU'lar zaten konfigüre edilmiş, memory growth ayarlanamıyor")
                print(f"🎮 Logical GPU sayısı: {len(logical_gpus)}")
                print(f"🎮 Physical GPU sayısı: {len(gpus)}")
            else:
                # Her GPU için memory growth etkinleştir
                for i, gpu in enumerate(gpus):
                    try:
                        tf.config.experimental.set_memory_growth(gpu, True)
                        print(f"✅ GPU {i} ({gpu.name}) için memory growth etkinleştirildi")
                    except Exception as e:
                        print(f"⚠️ GPU {i} memory growth hatası: {e}")
            
            # Float32 policy'yi zorla ayarla
            try:
                tf.keras.mixed_precision.set_global_policy('float32')
                print("✅ Float32 precision policy ayarlandı")
            except Exception as e:
                print(f"⚠️ Float32 policy hatası: {e}")
            
            for i, gpu in enumerate(gpus):
                try:
                    gpu_details = tf.config.experimental.get_device_details(gpu)
                    device_name = gpu_details.get('device_name', 'Unknown')
                    print(f"🎮 GPU {i}: {device_name}")
                except Exception as e:
                    print(f"🎮 GPU {i}: Detaylar alınamadı ({e})")
            
        except Exception as e:
            print(f"⚠️ GPU konfigürasyonu genel hatası: {e}")
            print("💡 GPU mevcut ama konfigürasyon yapılamadı, yine de kullanılabilir...")
            
    else:
        print("⚠️ GPU bulunamadı, CPU kullanılacak...")
    
    # TensorFlow GPU kullanımını doğrula
    try:
        gpu_available = len(tf.config.list_physical_devices('GPU')) > 0
        print(f"🔍 GPU kullanılabilir: {gpu_available}")
        
        logical_devices = tf.config.list_logical_devices()
        device_names = [device.name for device in logical_devices]
        print(f"🔍 Kullanılabilir cihazlar: {device_names}")
        
    except Exception as e:
        print(f"⚠️ GPU doğrulama hatası: {e}")

try:
    configure_gpu()
except Exception as e:
    print(f"❌ GPU konfigürasyonu başarısız: {e}")
    print("💡 CPU ile devam edilecek...")

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

# SelfAttention katmanını yeniden tanımlama 
class SelfAttention(Layer):
    def __init__(self, attention_units=128, return_attention=False, **kwargs):
        self.attention_units = attention_units
        self.return_attention = return_attention
        super(SelfAttention, self).__init__(**kwargs)
        
    def build(self, input_shape):
        self.time_steps = input_shape[1]
        self.input_dim = input_shape[2]
        
        from tensorflow.keras.layers import Dense
        self.query_dense = Dense(self.attention_units)
        self.key_dense = Dense(self.attention_units)
        self.value_dense = Dense(self.input_dim)
        self.context_dense = Dense(self.input_dim)
        
        super(SelfAttention, self).build(input_shape)
    
    def call(self, inputs):
        # Mixed precision için dtype kontrolü
        input_dtype = inputs.dtype
        
        query = self.query_dense(inputs)
        key = self.key_dense(inputs)
        value = self.value_dense(inputs)
        
        score = tf.matmul(query, key, transpose_b=True)
        
        # Mixed precision uyumlu sqrt hesaplama
        attention_units_tensor = tf.cast(self.attention_units, input_dtype)
        sqrt_attention_units = tf.math.sqrt(attention_units_tensor)
        score = score / sqrt_attention_units
        
        attention_weights = tf.nn.softmax(score, axis=-1)
        context = tf.matmul(attention_weights, value)
        output = self.context_dense(context)
        
        if self.return_attention:
            return output, attention_weights
        return output
    
    def compute_output_shape(self, input_shape):
        if self.return_attention:
            return [(input_shape[0], input_shape[1], self.input_dim), 
                    (input_shape[0], input_shape[1], input_shape[1])]
        return (input_shape[0], input_shape[1], self.input_dim)
    
    def get_config(self):
        config = super(SelfAttention, self).get_config()
        config.update({
            'attention_units': self.attention_units,
            'return_attention': self.return_attention
        })
        return config

def custom_cost_metric(y_true, y_pred):
    y_pred = tf.nn.softmax(y_pred)
    pred_class = tf.argmax(y_pred, axis=1)
    true_class = tf.argmax(y_true, axis=1)
    
    cost_matrix = tf.constant([
        [0, 7, 8, 9, 10],
        [200, 0, 7, 8, 9],
        [300, 200, 0, 7, 8],
        [400, 300, 200, 0, 7],
        [500, 400, 300, 200, 0]
    ], dtype=tf.float32)
    
    costs = tf.gather_nd(cost_matrix, 
                        tf.stack([true_class, pred_class], axis=1))
    return tf.reduce_mean(costs)

class TimeSeriesSHAPAnalyzer:
    def __init__(self, model_dir, model_name="best_cost_model.keras", output_dir="shap_results"):
        """
        Time Series Bi-LSTM Attention modeli için SHAP analizi yapan sınıf
        
        Args:
            model_dir: Model ve scaler'ların bulunduğu dizin
            model_name: Yüklenecek model dosyasının adı
            output_dir: SHAP görselleştirmelerinin kaydedileceği klasör
        """
        self.model_dir = model_dir
        self.model_name = model_name
        self.output_dir = output_dir
        self.model = None
        self.scaler = None
        self.label_encoder = None
        self.class_names = None
        self.feature_names = None
        self.model_input_shape = None 
        
        os.makedirs(self.output_dir, exist_ok=True)
        print(f"SHAP sonuçları şu klasöre kaydedilecek: {self.output_dir}")
        
        self.monitor_gpu_memory()
        
    def monitor_gpu_memory(self):
        """GPU memory kullanımını monitör et"""
        gpus = tf.config.experimental.list_physical_devices('GPU')
        if gpus:
            try:
                gpu_details = tf.config.experimental.get_device_details(gpus[0])
                print(f"🎮 Aktif GPU: {gpu_details.get('device_name', 'Unknown')}")
            except:
                print("🎮 GPU detayları alınamadı")
        
    def _get_save_path(self, filename):
        """Save path oluşturucu yardımcı fonksiyon"""
        if filename:
            return os.path.join(self.output_dir, filename)
        return None
        
    def load_model_and_preprocessors(self):
        """Model ve ön işleme araçlarını yükle"""
        custom_objects = {
            'SelfAttention': SelfAttention,
            'custom_cost_metric': custom_cost_metric
        }
        
        print("🚀 Model GPU'ya yükleniyor...")
        
        # Model yükleme - GPU'da - Mixed precision güvenli yaklaşım
        model_path = os.path.join(self.model_dir, self.model_name)
        
        try:
            with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
                self.model = load_model(model_path, custom_objects=custom_objects)
                print(f"✅ Model GPU'ya yüklendi: {model_path}")
        except Exception as e:
            print(f"⚠️ GPU model yükleme hatası: {e}")
            print("🔄 Mixed precision kapatılıp tekrar denenecek...")
            
            # Mixed precision'ı geçici olarak kapat
            original_policy = tf.keras.mixed_precision.global_policy()
            try:
                tf.keras.mixed_precision.set_global_policy('float32')
                with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
                    self.model = load_model(model_path, custom_objects=custom_objects)
                    print(f"✅ Model float32 ile GPU'ya yüklendi: {model_path}")
                # Policy'yi geri yükle
                tf.keras.mixed_precision.set_global_policy(original_policy)
            except Exception as e2:
                print(f"❌ GPU model yükleme tamamen başarısız: {e2}")
                print("🔄 CPU ile deneniyor...")
                tf.keras.mixed_precision.set_global_policy('float32')
                self.model = load_model(model_path, custom_objects=custom_objects)
                print(f"✅ Model CPU'ya yüklendi: {model_path}")
        
        # Model input shape'ini öğren
        if self.model:
            self.model_input_shape = self.model.input_shape
            print(f"🔍 Model input shape: {self.model_input_shape}")
            
        # Scaler yükleme
        scaler_path = os.path.join(self.model_dir, 'standard_scaler.joblib')
        self.scaler = joblib.load(scaler_path)
        print(f"✅ Scaler yüklendi: {scaler_path}")
        
        # Label encoder yükleme
        le_path = os.path.join(self.model_dir, 'label_encoder.joblib')
        self.label_encoder = joblib.load(le_path)
        self.class_names = self.label_encoder.classes_
        print(f"✅ Label encoder yüklendi: {le_path}")
        print(f"✅ Sınıflar: {self.class_names}")
        
    def _verify_model_device(self):
        """Model'in hangi cihazda olduğunu doğrula - doğru input shape ile"""
        try:
            if self.model_input_shape and len(self.model_input_shape) >= 3:
                expected_features = self.model_input_shape[-1]  # Son boyut feature sayısı
                expected_timesteps = self.model_input_shape[1] if self.model_input_shape[1] else 10  # None ise 10 varsayalım
                
                with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
                    dummy_input = tf.random.normal((1, expected_timesteps, expected_features))
                    _ = self.model(dummy_input, training=False)
                    device = '/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'
                    print(f"✅ Model {device} üzerinde çalışıyor")
                    print(f"✅ Test input shape: (1, {expected_timesteps}, {expected_features})")
            else:
                print("⚠️ Model input shape belirlenemedi, device test atlanıyor")
        except Exception as e:
            print(f"⚠️ Model device doğrulama hatası: {e}")
        
    def prepare_data_for_shap(self, data_file, n_background=100, n_explain=50):
        """
        SHAP analizi için veri hazırlama
        """
        print("📊 GPU optimizasyonu ile veri hazırlanıyor...")
        
        data = pd.read_csv(data_file)
        
        # Feature isimlerini al (vehicle_id ve time_step hariç)
        feature_cols = [col for col in data.columns if col not in ['vehicle_id', 'time_step', 'class_labels']]
        self.feature_names = feature_cols
        print(f"🔍 Feature sayısı: {len(self.feature_names)}")
        
        # Model'in beklediği feature sayısı ile kontrol et
        if self.model_input_shape and len(self.model_input_shape) >= 3:
            expected_features = self.model_input_shape[-1]
            if len(self.feature_names) != expected_features:
                print(f"⚠️ Veri feature sayısı ({len(self.feature_names)}) model'in beklediği sayı ({expected_features}) ile uyuşmuyor!")
                print(f"⚠️ Ilk {expected_features} feature kullanılacak...")
                feature_cols = feature_cols[:expected_features]
                self.feature_names = feature_cols
        
        # Her vehicle_id için zaman serisi hazırlama
        vehicle_groups = data.groupby('vehicle_id')
        
        sequences = []
        vehicle_ids = []
        original_lengths = [] 
        
        print(f"🚗 Toplam vehicle sayısı: {len(vehicle_groups)}")
        
        for vehicle_id, group in vehicle_groups:
            # Özellik sütunlarını al
            time_series = group.sort_values('time_step')[feature_cols].values
            
            # Orijinal uzunluğu kaydet (padding öncesi)
            original_lengths.append(len(time_series))
            
            # Normalizasyon
            time_series = self.scaler.transform(time_series)
            
            sequences.append(time_series)
            vehicle_ids.append(vehicle_id)
        
        # Sequences'ı aynı uzunluğa getirme
        max_length = max(len(seq) for seq in sequences)
        print(f"⏱️ Maksimum sequence uzunluğu: {max_length}")
        print(f"📏 Ortalama gerçek uzunluk: {np.mean(original_lengths):.1f}")
        print(f"📏 Min-Max gerçek uzunluk: {min(original_lengths)}-{max(original_lengths)}")
        
        padded_sequences = []
        for i, seq in enumerate(sequences):
            if len(seq) < max_length:
                pad_width = ((0, max_length - len(seq)), (0, 0))
                padded_seq = np.pad(seq, pad_width, mode='constant', constant_values=0)
            else:
                padded_seq = seq
            padded_sequences.append(padded_seq)
        
        X = np.array(padded_sequences, dtype=np.float32)  # GPU için float32

        print("🎯 Background data seçimi...")
        
        # Tüm veriyi farklı uzunluk gruplarına ayır
        length_groups = {}
        for i, length in enumerate(original_lengths):
            length_bucket = (length // 10) * 10  # 10'luk gruplara ayır
            if length_bucket not in length_groups:
                length_groups[length_bucket] = []
            length_groups[length_bucket].append(i)
        
        print(f"📊 Uzunluk grupları: {[(k, len(v)) for k, v in length_groups.items()]}")
        
        # Her uzunluk grubundan proportional sampling
        background_indices = []
        explain_indices = []
        
        total_samples = min(n_background + n_explain, len(X))
        
        for length_bucket, indices in length_groups.items():
            group_size = len(indices)
            
            # Bu gruptan alınacak sample sayıları (proportional)
            group_bg_count = int(n_background * group_size / len(X))
            group_ex_count = int(n_explain * group_size / len(X))
            
            # En az 1 sample al (eğer grup varsa)
            group_bg_count = max(1, min(group_bg_count, group_size - 1)) if group_size > 1 else min(1, group_size)
            group_ex_count = max(1, min(group_ex_count, group_size - group_bg_count)) if group_size > group_bg_count else 0
            
            # Random seçim
            group_indices = np.random.choice(indices, 
                                           min(group_bg_count + group_ex_count, group_size), 
                                           replace=False)
            
            background_indices.extend(group_indices[:group_bg_count])
            if group_ex_count > 0:
                explain_indices.extend(group_indices[group_bg_count:group_bg_count + group_ex_count])
            
            print(f"  Uzunluk {length_bucket}: {group_bg_count} background, {group_ex_count} explain")
        
        # Eksik sample'ları tamamla
        remaining_indices = [i for i in range(len(X)) if i not in background_indices and i not in explain_indices]
        
        if len(background_indices) < n_background and remaining_indices:
            needed = n_background - len(background_indices)
            additional = np.random.choice(remaining_indices, min(needed, len(remaining_indices)), replace=False)
            background_indices.extend(additional)
            remaining_indices = [i for i in remaining_indices if i not in additional]
        
        if len(explain_indices) < n_explain and remaining_indices:
            needed = n_explain - len(explain_indices)
            additional = np.random.choice(remaining_indices, min(needed, len(remaining_indices)), replace=False)
            explain_indices.extend(additional)
        
        # Final selection
        self.X_background = X[background_indices]
        self.X_explain = X[explain_indices]
        
        # Gerçek uzunlukları da kaydet
        self.background_real_lengths = [original_lengths[i] for i in background_indices]
        self.explain_real_lengths = [original_lengths[i] for i in explain_indices]
        
        # GPU tensörlerine dönüştür
        with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
            self.X_background_gpu = tf.constant(self.X_background, dtype=tf.float32)
            self.X_explain_gpu = tf.constant(self.X_explain, dtype=tf.float32)
        
        print(f"✅ Background data shape: {self.X_background.shape}")
        print(f"✅ Explain data shape: {self.X_explain.shape}")
        print(f"📈 Background gerçek uzunluklar - min: {min(self.background_real_lengths)}, max: {max(self.background_real_lengths)}, avg: {np.mean(self.background_real_lengths):.1f}")
        print(f"📈 Explain gerçek uzunluklar - min: {min(self.explain_real_lengths)}, max: {max(self.explain_real_lengths)}, avg: {np.mean(self.explain_real_lengths):.1f}")
        print(f"📈 Toplam veri kullanımı: {len(background_indices) + len(explain_indices)}/{len(X)} vehicle (%{100*(len(background_indices) + len(explain_indices))/len(X):.1f})")
        print(f"🚀 Veriler GPU'ya yüklendi")
        
        # Model device test - artık doğru feature sayısı ile
        self._verify_model_device()
        
        return self.X_background, self.X_explain
    
    def create_flattened_model_wrapper(self):
        """
        3D time series verisini 2D'ye düzleştirip tekrar şekillendiren model wrapper - GPU optimizeli
        """
        @tf.function  # GPU performansı için
        def predict_function(X_flat):
            with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
                # X_flat shape: (n_samples, n_timesteps * n_features)
                n_samples = tf.shape(X_flat)[0]
                n_timesteps = self.X_background.shape[1]
                n_features = self.X_background.shape[2]
                
                # 3D'ye yeniden şekillendir
                X_reshaped = tf.reshape(X_flat, [n_samples, n_timesteps, n_features])
                
                # Model tahminini al
                predictions = self.model(X_reshaped, training=False)
                
                # Softmax uygula
                predictions = tf.nn.softmax(predictions)
                
                return predictions
        
        return predict_function
    
    def create_model_wrapper(self):
        """Model için wrapper oluştur (SHAP uyumluluğu için) - GPU optimizeli"""
        @tf.function  # GPU performansı için
        def model_call(x):
            with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
                return self.model(x, training=False)
        return model_call
    
    def run_shap_analysis(self, method='masking'):
        """
        Time series için çoklu SHAP analizi yöntemi - GPU optimizeli
        
        Args:
            method: 'deep', 'gradient', 'masking' veya 'sampling'
        """
        if self.model is None:
            raise ValueError("Önce load_model_and_preprocessors() çağırın")
            
        if not hasattr(self, 'X_background'):
            raise ValueError("Önce prepare_data_for_shap() çağırın")
        
        print(f"🚀 GPU ile Time Series SHAP analizi başlatılıyor ({method} yöntemi)...")
        
        n_timesteps = self.X_background.shape[1]
        n_features = self.X_background.shape[2]
        
        if method == 'masking':
            try:
                print("🔥 GPU Masking tabanlı SHAP analizi...")
                # Time series için özel masking yaklaşımı
                self._run_masking_shap()
                return
                
            except Exception as e:
                print(f"❌ Masking yöntemi hatası: {e}")
                print("🔄 Sampling yöntemi deneniyor...")
        
        if method == 'sampling' or method == 'masking':
            try:
                print("🔥 GPU Sampling tabanlı SHAP analizi...")
                # Basit sampling yaklaşımı
                self._run_sampling_shap()
                return
                
            except Exception as e:
                print(f"❌ Sampling yöntemi hatası: {e}")
                raise ValueError("❌ Hiçbir SHAP yöntemi çalışmadı!")
    
    def _run_masking_shap(self, use_all_timesteps=True, use_feature_masking=True):

        n_samples = self.X_explain.shape[0]
        n_timesteps = self.X_explain.shape[1]
        n_features = self.X_explain.shape[2]
        n_classes = len(self.class_names)
        
        print(f"🚀 GPU masking analizi başlatılıyor...")
        print(f"- Samples: {n_samples}")
        print(f"- Time steps: {n_timesteps}")
        print(f"- Features: {n_features}")
        print(f"- Classes: {n_classes}")
        
        # GPU'da multiple baseline'lar oluştur (DÜZELTME: TensorFlow indexing)
        with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
            n_baselines = min(10, self.X_background.shape[0])
            baseline_indices = np.random.choice(self.X_background.shape[0], n_baselines, replace=False)
            
            # NumPy array'i TensorFlow tensor'a çevir
            baseline_indices_tf = tf.constant(baseline_indices, dtype=tf.int32)
            baselines = tf.gather(self.X_background_gpu, baseline_indices_tf) 
            
            # Her baseline için tahmin al
            baseline_preds = []
            for i in range(n_baselines):
                baseline_pred = self.model(baselines[i:i+1], training=False)
                baseline_pred = tf.nn.softmax(baseline_pred)
                baseline_preds.append(baseline_pred)
            baseline_preds = tf.concat(baseline_preds, axis=0)  # (n_baselines, n_classes)
            mean_baseline_pred = tf.reduce_mean(baseline_preds, axis=0, keepdims=True)  # (1, n_classes)
        
        # Gerçek sequence uzunluklarını bul (padding etkisini çıkar)
        real_sequence_lengths = []
        for sample_idx in range(n_samples):
            sample = self.X_explain[sample_idx]
            # Son sıfır olmayan timestep'i bul (padding detection)
            non_zero_timesteps = np.where(np.any(sample != 0, axis=1))[0]
            if len(non_zero_timesteps) > 0:
                real_length = non_zero_timesteps[-1] + 1
            else:
                real_length = n_timesteps  # Eğer hiç sıfır yoksa full length
            real_sequence_lengths.append(min(real_length, n_timesteps))
        
        print(f"📏 Gerçek sequence uzunlukları: min={min(real_sequence_lengths)}, max={max(real_sequence_lengths)}, avg={np.mean(real_sequence_lengths):.1f}")
        
        # Her örnek için SHAP değerleri
        all_shap_values = []
        
        for sample_idx in range(n_samples):
            print(f"\n=== GPU Sample {sample_idx + 1}/{n_samples} analiz ediliyor ===")
            real_length = real_sequence_lengths[sample_idx]
            
            with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
                sample = self.X_explain_gpu[sample_idx:sample_idx+1]
                sample_pred = self.model(sample, training=False)
                sample_pred = tf.nn.softmax(sample_pred)
            
            # SHAP değerleri için matrix
            feature_contributions = np.zeros((n_timesteps, n_features, n_classes))
            
            # Sadece gerçek sequence içindeki timestep'leri işle
            if use_all_timesteps:
                timesteps_to_process = list(range(real_length))  # Padding'siz
                print(f"🔥 GPU ile gerçek {real_length}/{n_timesteps} time step işlenecek...")
            else:
                timesteps_to_process = list(range(0, real_length, max(1, real_length // 20)))
                print(f"🔥 GPU ile {len(timesteps_to_process)} time step sampling ile işlenecek...")
            
            total_operations = len(timesteps_to_process)
            if use_feature_masking:
                total_operations *= n_features
            
            operation_count = 0
            batch_size = 16 if use_feature_masking else 32
            
            for t_idx, t in enumerate(timesteps_to_process):
                if t_idx % max(1, len(timesteps_to_process) // 5) == 0:
                    progress = (t_idx / len(timesteps_to_process)) * 100
                    print(f"  ⚡ GPU Time step progress: {progress:.1f}% ({t_idx}/{len(timesteps_to_process)})")
                
                if use_feature_masking:
                    # Koalisyon tabanlı masking stratejisi (TensorFlow uyumlu)
                    with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
                        for f_start in range(0, n_features, batch_size):
                            f_end = min(f_start + batch_size, n_features)
                            batch_features = list(range(f_start, f_end))
                            
                            # Batch için koalisyon sampling
                            masked_samples = []
                            coalition_samples = []
                            
                            for f in batch_features:
                                # Random baseline seçimi 
                                baseline_idx = np.random.randint(0, n_baselines)
                                current_baseline = baselines[baseline_idx:baseline_idx+1]
                                current_baseline_pred = baseline_preds[baseline_idx:baseline_idx+1]
                                
                                # Feature'ı mask et
                                masked_sample = tf.identity(sample)
                                indices = tf.constant([[0, t, f]], dtype=tf.int32)
                                updates = tf.expand_dims(current_baseline[0, t, f], 0)
                                masked_sample = tf.tensor_scatter_nd_update(
                                    masked_sample, indices, updates
                                )
                                masked_samples.append(masked_sample)
                                
                                # Koalisyon sample'ı (feature hariç diğerleri baseline)
                                coalition_sample = tf.identity(current_baseline)
                                indices = tf.constant([[0, t, f]], dtype=tf.int32)
                                updates = tf.expand_dims(sample[0, t, f], 0)
                                coalition_sample = tf.tensor_scatter_nd_update(
                                    coalition_sample, indices, updates
                                )
                                coalition_samples.append((coalition_sample, current_baseline_pred))
                            
                            if masked_samples:
                                # Batch tahmin
                                masked_batch = tf.concat(masked_samples, axis=0)
                                masked_pred_batch = self.model(masked_batch, training=False)
                                masked_pred_batch = tf.nn.softmax(masked_pred_batch)
                                
                                # SHAP formülü
                                for i, f in enumerate(batch_features):
                                    coalition_sample, coalition_baseline_pred = coalition_samples[i]
                                    
                                    # Koalisyon tahmini
                                    coalition_pred = self.model(coalition_sample, training=False)
                                    coalition_pred = tf.nn.softmax(coalition_pred)
                                    
                                    # Marginal contribution (SHAP formülü)
                                    # SHAP = f(S ∪ {i}) - f(S) burada S koalisyon, i feature
                                    with_feature = sample_pred - mean_baseline_pred
                                    without_feature = masked_pred_batch[i:i+1] - coalition_baseline_pred
                                    
                                    contribution = with_feature - without_feature
                                    feature_contributions[t, f, :] = contribution.numpy()[0]
                                    operation_count += 1
                else:
                    # Tüm time step'i mask et (koalisyon yaklaşımı ile)
                    with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
                        baseline_idx = np.random.randint(0, n_baselines)
                        current_baseline = baselines[baseline_idx:baseline_idx+1]
                        current_baseline_pred = baseline_preds[baseline_idx:baseline_idx+1]
                        
                        masked_sample = tf.identity(sample)
                        indices = tf.constant([[0, t, f] for f in range(n_features)], dtype=tf.int32)
                        updates = tf.stack([current_baseline[0, t, f] for f in range(n_features)])
                        masked_sample = tf.tensor_scatter_nd_update(
                            masked_sample, indices, updates
                        )
                        
                        masked_pred = self.model(masked_sample, training=False)
                        masked_pred = tf.nn.softmax(masked_pred)
                        
                        # SHAP formülü ile katkıyı hesapla
                        with_timestep = sample_pred - mean_baseline_pred
                        without_timestep = masked_pred - current_baseline_pred
                        
                        contribution = with_timestep - without_timestep
                        contribution_per_feature = contribution.numpy()[0] / n_features
                        for f in range(n_features):
                            feature_contributions[t, f, :] = contribution_per_feature
                        
                        operation_count += 1
            
            all_shap_values.append(feature_contributions)
            print(f"✅ GPU Sample {sample_idx + 1} tamamlandı! ({operation_count} operasyon)")
        
        self.shap_values = np.array(all_shap_values)  # (n_samples, n_timesteps, n_features, n_classes)
        # Sınıf eksenini ilk eksene taşı - SADECE BİR KEZ
        self.shap_values = np.transpose(self.shap_values, (3, 0, 1, 2))  # (n_classes, n_samples, n_timesteps, n_features)
        
        print(f"\n🎉 GPU Masking SHAP tamamlandı!")
        print(f"✅ SHAP shape: {self.shap_values.shape}")
        print(f"⚡ Toplam GPU operasyon: {total_operations * n_samples}")
        
        # İstatistikler
        shap_abs_mean = np.abs(self.shap_values).mean()
        shap_abs_max = np.abs(self.shap_values).max()
        print(f"📊 SHAP değerleri - Ortalama: {shap_abs_mean:.6f}, Maksimum: {shap_abs_max:.6f}")
        
        # SHAP değerlerinin sıfır toplamını kontrol et (debugging)
        for class_idx in range(n_classes):
            class_sum = np.sum(self.shap_values[class_idx], axis=(1, 2))  # Her sample için toplam
            print(f"Class {class_idx} SHAP sum check - Mean: {np.mean(class_sum):.6f}, Std: {np.std(class_sum):.6f}")
    
    def _run_sampling_shap(self):

        n_samples = self.X_explain.shape[0]
        n_timesteps = self.X_explain.shape[1]
        n_features = self.X_explain.shape[2]
        n_classes = len(self.class_names)
        
        print("🚀 GPU Sampling SHAP başlatılıyor...")
        
        # Multiple baseline'lar 
        with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
            n_baselines = min(20, self.X_background.shape[0])
            baseline_indices = np.random.choice(self.X_background.shape[0], n_baselines, replace=False)
            
            baseline_indices_tf = tf.constant(baseline_indices, dtype=tf.int32)
            baselines = tf.gather(self.X_background_gpu, baseline_indices_tf)  
            
            # Baseline tahminleri
            baseline_preds = []
            for i in range(n_baselines):
                baseline_pred = self.model(baselines[i:i+1], training=False)
                baseline_pred = tf.nn.softmax(baseline_pred)
                baseline_preds.append(baseline_pred[0])
            baseline_preds = tf.stack(baseline_preds)  # (n_baselines, n_classes)
            mean_baseline_pred = tf.reduce_mean(baseline_preds, axis=0)
        
        all_shap_values = []
        
        for sample_idx in range(n_samples):
            print(f"⚡ GPU Sample {sample_idx + 1}/{n_samples} sampling analizi...")
            
            # Gerçek sequence uzunluğunu bul
            sample_np = self.X_explain[sample_idx]
            non_zero_timesteps = np.where(np.any(sample_np != 0, axis=1))[0]
            if len(non_zero_timesteps) > 0:
                real_length = non_zero_timesteps[-1] + 1
            else:
                real_length = n_timesteps
            real_length = min(real_length, n_timesteps)
            
            with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
                sample = self.X_explain_gpu[sample_idx]
                sample_pred = self.model(sample[tf.newaxis, ...], training=False)
                sample_pred = tf.nn.softmax(sample_pred)[0]
                
                feature_attributions = np.zeros((n_timesteps, n_features, n_classes))
                
                # SHAP hesaplama 
                # Monte Carlo sampling ile Shapley değerlerini yaklaşık hesapla
                n_samples_per_feature = 1000  # Her feature için sampling sayısı
                
                for t in range(real_length):  # Sadece gerçek timestep'ler
                    if t % max(1, real_length // 10) == 0:
                        progress = (t / real_length) * 100
                        print(f"    Timestep progress: {progress:.1f}%")
                    
                    for f in range(n_features):
                        # Bu feature için Shapley değerini hesapla
                        marginal_contributions = []
                        
                        for _ in range(n_samples_per_feature):
                            # Random koalisyon oluştur (bu feature hariç)
                            coalition_size = np.random.randint(0, n_features - 1)
                            other_features = [idx for idx in range(n_features) if idx != f]
                            
                            if len(other_features) == 0:
                                coalition_features = []
                            else:
                                coalition_features = np.random.choice(other_features, 
                                                                    min(coalition_size, len(other_features)), 
                                                                    replace=False)
                            
                            # Random baseline seç
                            baseline_idx = np.random.randint(0, n_baselines)
                            current_baseline = baselines[baseline_idx]
                            current_baseline_pred = baseline_preds[baseline_idx]
                            
                            # Koalisyon + target feature sample oluştur
                            coalition_with_target = tf.identity(current_baseline[tf.newaxis, ...])
                            # Koalisyon feature'larını sample'dan al
                            for cf in coalition_features:
                                indices = tf.constant([[0, t, cf]], dtype=tf.int32)
                                updates = tf.expand_dims(sample[t, cf], 0)
                                coalition_with_target = tf.tensor_scatter_nd_update(
                                    coalition_with_target, indices, updates
                                )
                            # Target feature'ı ekle
                            indices = tf.constant([[0, t, f]], dtype=tf.int32)
                            updates = tf.expand_dims(sample[t, f], 0)
                            coalition_with_target = tf.tensor_scatter_nd_update(
                                coalition_with_target, indices, updates
                            )
                            
                            # Koalisyon (target feature olmadan) sample oluştur
                            coalition_without_target = tf.identity(current_baseline[tf.newaxis, ...])
                            for cf in coalition_features:
                                indices = tf.constant([[0, t, cf]], dtype=tf.int32)
                                updates = tf.expand_dims(sample[t, cf], 0)
                                coalition_without_target = tf.tensor_scatter_nd_update(
                                    coalition_without_target, indices, updates
                                )
                            
                            # Tahminleri al
                            with_target_pred = self.model(coalition_with_target, training=False)
                            with_target_pred = tf.nn.softmax(with_target_pred)[0]
                            
                            without_target_pred = self.model(coalition_without_target, training=False)
                            without_target_pred = tf.nn.softmax(without_target_pred)[0]
                            
                            # Marginal contribution
                            marginal_contrib = with_target_pred - without_target_pred
                            marginal_contributions.append(marginal_contrib.numpy())
                        
                        # Bu feature'ın ortalama Shapley değeri
                        feature_shap = np.mean(marginal_contributions, axis=0)
                        feature_attributions[t, f, :] = feature_shap
            
            all_shap_values.append(feature_attributions)
            print(f"✅ Sample {sample_idx + 1} tamamlandı (gerçek length: {real_length})")
        
        # Transpose sadece bir kez
        self.shap_values = np.array(all_shap_values)  # (n_samples, n_timesteps, n_features, n_classes)
        self.shap_values = np.transpose(self.shap_values, (3, 0, 1, 2))  # (n_classes, n_samples, n_timesteps, n_features)
        
        print(f"✅ GPU Sampling SHAP shape: {self.shap_values.shape}")
        print("🎉 GPU Sampling tabanlı SHAP tamamlandı!")
        
        # Shapley değerlerinin toplamının sıfıra yakın olduğunu kontrol et
        for class_idx in range(n_classes):
            class_shap = self.shap_values[class_idx]
            sample_sums = np.sum(class_shap, axis=(1, 2))  # Her sample için toplam
            print(f"Class {class_idx} Shapley sum check - Mean: {np.mean(sample_sums):.6f}, Std: {np.std(sample_sums):.6f}")
            print(f"  (Shapley değerleri sıfıra yakın toplanmalı)")
        
    def plot_feature_importance_over_time(self, sample_idx=0, class_idx=0, top_features=10, 
                                        figsize=(15, 10), save_path=None):
        """
        Time series için özellik önemini zaman boyutunda görselleştir
        """
        if not hasattr(self, 'shap_values'):
            raise ValueError("Önce run_shap_analysis() çağırın")
        
        # SHAP değerlerini al
        if len(self.shap_values.shape) == 4:  # Multi-class
            sample_shap = self.shap_values[class_idx, sample_idx]  # (n_timesteps, n_features)
            class_name = self.class_names[class_idx]
        else:
            sample_shap = self.shap_values[sample_idx]  # (n_timesteps, n_features)
            class_name = "Output"
        
        # Gerçek sequence uzunluğunu al
        if hasattr(self, 'explain_real_lengths') and sample_idx < len(self.explain_real_lengths):
            real_length = self.explain_real_lengths[sample_idx]
        else:
            # Fallback: sıfır olmayan timestep'leri bul
            sample_data = self.X_explain[sample_idx]
            non_zero_timesteps = np.where(np.any(sample_data != 0, axis=1))[0]
            real_length = non_zero_timesteps[-1] + 1 if len(non_zero_timesteps) > 0 else sample_shap.shape[0]
        
        real_length = min(real_length, sample_shap.shape[0])
        print(f"📏 Sample {sample_idx} gerçek uzunluk: {real_length}/{sample_shap.shape[0]}")
        
        # Sadece gerçek timestep'leri kullan
        sample_shap_real = sample_shap[:real_length, :]  # (real_length, n_features)
        
        # 1. Basit toplam
        simple_importance = np.abs(sample_shap_real).sum(axis=0)
        
        # 2. Ağırlıklı temporal importance (zaman yakın olan daha önemli)
        time_weights = np.exp(-0.1 * np.arange(real_length))  # Exponential decay
        weighted_importance = np.sum(np.abs(sample_shap_real) * time_weights[:, np.newaxis], axis=0)
        
        # 3. Variance-based importance (temporal variation'ı ölçer)
        variance_importance = np.var(sample_shap_real, axis=0)
        
        # 4. Peak-based importance (maksimum katkıyı ölçer)
        peak_importance = np.max(np.abs(sample_shap_real), axis=0)
        
        # 5. Consistency-based importance (sürekli katkı)
        non_zero_ratio = np.mean(np.abs(sample_shap_real) > 1e-6, axis=0)  # Sıfır olmayan timestep oranı
        consistency_importance = simple_importance * non_zero_ratio
        
        # Kombinasyon importance (weighted average)
        combined_importance = (
            0.3 * simple_importance / np.max(simple_importance) + 
            0.2 * weighted_importance / np.max(weighted_importance) + 
            0.2 * variance_importance / np.max(variance_importance) + 
            0.15 * peak_importance / np.max(peak_importance) + 
            0.15 * consistency_importance / np.max(consistency_importance)
        )
        
        # En önemli özellikleri bul (combined importance'a göre)
        top_feature_indices = np.argsort(combined_importance)[-top_features:]
        
        fig, axes = plt.subplots(4, 1, figsize=figsize)
        
        # 1. Zaman boyutunda SHAP değerleri (heatmap) - sadece gerçek timestep'ler
        sample_shap_top = sample_shap_real[:, top_feature_indices]
        
        im = axes[0].imshow(sample_shap_top.T, aspect='auto', cmap='RdBu_r', 
                           vmin=-np.abs(sample_shap_top).max(), 
                           vmax=np.abs(sample_shap_top).max())
        
        axes[0].set_xlabel('Time Steps (Gerçek Veriler)')
        axes[0].set_ylabel('Features')
        axes[0].set_title(f'SHAP Values Over Time - Sample {sample_idx} - Class: {class_name}\n(Gerçek uzunluk: {real_length}, Padding çıkarıldı)')
        
        # Feature isimlerini y-axis'e ekle
        if self.feature_names:
            feature_labels = [self.feature_names[idx] for idx in top_feature_indices]
        else:
            feature_labels = [f'Feature_{idx}' for idx in top_feature_indices]
        
        axes[0].set_yticks(range(len(top_feature_indices)))
        axes[0].set_yticklabels(feature_labels, fontsize=8)
        
        # Gerçek uzunluğu vurgula
        axes[0].axvline(x=real_length-0.5, color='red', linestyle='--', alpha=0.7, label='Gerçek uzunluk sonu')
        axes[0].legend()
        
        # Colorbar ekle
        plt.colorbar(im, ax=axes[0], label='SHAP Value')
        
        # 2. Çoklu importance metriklerini karşılaştır
        x_pos = np.arange(len(top_feature_indices))
        width = 0.15
        
        axes[1].bar(x_pos - 2*width, simple_importance[top_feature_indices], width, 
                   label='Simple Sum', alpha=0.8, color='skyblue')
        axes[1].bar(x_pos - width, weighted_importance[top_feature_indices], width, 
                   label='Time Weighted', alpha=0.8, color='lightgreen')
        axes[1].bar(x_pos, variance_importance[top_feature_indices], width, 
                   label='Temporal Variance', alpha=0.8, color='orange')
        axes[1].bar(x_pos + width, peak_importance[top_feature_indices], width, 
                   label='Peak Value', alpha=0.8, color='pink')
        axes[1].bar(x_pos + 2*width, combined_importance[top_feature_indices], width, 
                   label='Combined', alpha=0.8, color='red')
        
        axes[1].set_xticks(x_pos)
        axes[1].set_xticklabels(feature_labels, rotation=45, ha='right', fontsize=8)
        axes[1].set_ylabel('Importance Score')
        axes[1].set_title('Çoklu Feature Importance Metrikleri')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        # 3. Zaman boyutunda temporal patterns
        # Sliding window ile temporal importance hesapla
        window_size = max(3, real_length // 10)
        temporal_importance = []
        time_windows = []
        
        for start in range(0, real_length - window_size + 1, max(1, window_size // 2)):
            end = min(start + window_size, real_length)
            window_importance = np.abs(sample_shap_real[start:end, top_feature_indices]).mean()
            temporal_importance.append(window_importance)
            time_windows.append((start + end) / 2)  # Window center
        
        axes[2].plot(time_windows, temporal_importance, linewidth=2, color='red', marker='o')
        axes[2].set_xlabel('Time Steps (Window Centers)')
        axes[2].set_ylabel('Window-based Mean |SHAP|')
        axes[2].set_title(f'Temporal Importance Patterns (Window size: {window_size})')
        axes[2].grid(True, alpha=0.3)
        axes[2].axvline(x=real_length, color='red', linestyle='--', alpha=0.5, label='Gerçek uzunluk sonu')
        axes[2].legend()
        
        # 4. Feature'ların zaman boyutunda consistency analizi
        feature_consistency = []
        feature_names_short = []
        
        for i, feat_idx in enumerate(top_feature_indices[:8]):  # İlk 8 feature
            feat_shap = sample_shap_real[:, feat_idx]
            
            # Consistency metrics
            non_zero_count = np.sum(np.abs(feat_shap) > 1e-6)
            mean_abs = np.mean(np.abs(feat_shap))
            std_val = np.std(feat_shap)
            consistency_score = non_zero_count / real_length * mean_abs / (std_val + 1e-8)
            
            feature_consistency.append(consistency_score)
            feat_name = feature_labels[i] if i < len(feature_labels) else f'F_{feat_idx}'
            feature_names_short.append(feat_name[:15] + '...' if len(feat_name) > 15 else feat_name)
        
        bars = axes[3].bar(range(len(feature_consistency)), feature_consistency, 
                          color='lightcoral', alpha=0.7)
        axes[3].set_xticks(range(len(feature_consistency)))
        axes[3].set_xticklabels(feature_names_short, rotation=45, ha='right', fontsize=8)
        axes[3].set_ylabel('Consistency Score')
        axes[3].set_title('Feature Temporal Consistency (Non-zero ratio × Mean / Std)')
        axes[3].grid(True, alpha=0.3)
        
        # Consistency değerlerini bar'ların üstüne yaz
        for bar, score in zip(bars, feature_consistency):
            height = bar.get_height()
            axes[3].text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                        f'{score:.3f}', ha='center', va='bottom', fontsize=8)
        
        plt.tight_layout()
        
        # Kaydetme
        final_save_path = self._get_save_path(save_path) if save_path else None
        if final_save_path:
            plt.savefig(final_save_path, dpi=300, bbox_inches='tight')
            print(f"Plot kaydedildi: {final_save_path}")
        plt.show()
        
        # İstatistiksel özet yazdır
        print(f"\n📊 Feature Importance Özeti (Sample {sample_idx}, Class {class_name}):")
        print(f"📏 Analiz edilen timestep sayısı: {real_length} (padding çıkarıldı)")
        print(f"🔝 En önemli 5 feature (Combined importance):")
        for i, feat_idx in enumerate(top_feature_indices[-5:][::-1]):
            feat_name = self.feature_names[feat_idx] if self.feature_names else f"Feature_{feat_idx}"
            importance = combined_importance[feat_idx]
            consistency = feature_consistency[-(i+1)] if i < len(feature_consistency) else 0
            print(f"  {i+1}. {feat_name}: {importance:.6f} (consistency: {consistency:.6f})")

    def compute_corrected_shap_statistics(self):
        """
        SHAP değerlerinin istatistiksel doğrulaması
        
        Bu fonksiyon SHAP değerlerinin matematiksel olarak doğru olup olmadığını kontrol eder:
        1. Shapley değerlerinin toplamı sıfıra yakın olmalı (efficiency axiom)
        2. Baseline'dan sapma oranı kontrol edilir
        3. Temporal consistency kontrol edilir
        """
        if not hasattr(self, 'shap_values'):
            raise ValueError("Önce run_shap_analysis() çağırın")
        
        print("🔍 SHAP İstatistiksel Doğrulama Başlatılıyor...")
        
        n_classes, n_samples, n_timesteps, n_features = self.shap_values.shape
        
        # GPU'da baseline ve sample tahminlerini al
        with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
            baseline_mean = tf.reduce_mean(self.X_background_gpu, axis=0, keepdims=True)
            baseline_pred = self.model(baseline_mean, training=False)
            baseline_pred = tf.nn.softmax(baseline_pred)[0]
            
            sample_preds = []
            for i in range(n_samples):
                sample_pred = self.model(self.X_explain_gpu[i:i+1], training=False)
                sample_pred = tf.nn.softmax(sample_pred)[0]
                sample_preds.append(sample_pred)
            sample_preds = tf.stack(sample_preds)
        
        statistics = {}
        
        for class_idx in range(n_classes):
            class_name = self.class_names[class_idx]
            class_shap = self.shap_values[class_idx]  # (n_samples, n_timesteps, n_features)
            
            print(f"\n📊 Class {class_idx} ({class_name}) analizi:")
            
            # 1. Efficiency Axiom: SHAP değerleri toplamı = f(x) - f(baseline)
            shap_sums = np.sum(class_shap, axis=(1, 2))  # Her sample için toplam
            
            # Gerçek prediction farkları
            real_diffs = (sample_preds[:, class_idx] - baseline_pred[class_idx]).numpy()
            
            # Fark
            efficiency_errors = np.abs(shap_sums - real_diffs)
            
            print(f"  ✅ Efficiency Axiom:")
            print(f"     SHAP toplam ortalama: {np.mean(shap_sums):.6f}")
            print(f"     Gerçek fark ortalama: {np.mean(real_diffs):.6f}")
            print(f"     Mutlak hata ortalama: {np.mean(efficiency_errors):.6f}")
            print(f"     Maksimum hata: {np.max(efficiency_errors):.6f}")
            
            # 2. Temporal Consistency: Padding'li bölgelerde SHAP sıfır olmalı
            temporal_violations = []
            if hasattr(self, 'explain_real_lengths'):
                for sample_idx in range(n_samples):
                    real_length = self.explain_real_lengths[sample_idx]
                    if real_length < n_timesteps:
                        # Padding bölgesindeki SHAP değerleri
                        padding_shap = class_shap[sample_idx, real_length:, :]
                        padding_violation = np.sum(np.abs(padding_shap))
                        temporal_violations.append(padding_violation)
            
            if temporal_violations:
                print(f"  ✅ Temporal Consistency:")
                print(f"     Padding bölgesi SHAP ortalaması: {np.mean(temporal_violations):.6f}")
                print(f"     Maksimum padding violation: {np.max(temporal_violations):.6f}")
            
            # 3. Feature-wise Statistics
            feature_means = np.mean(np.abs(class_shap), axis=(0, 1))  # Her feature için ortalama
            feature_stds = np.std(class_shap, axis=(0, 1))
            
            # En önemli ve en az önemli feature'lar
            top_features_idx = np.argsort(feature_means)[-5:]
            bottom_features_idx = np.argsort(feature_means)[:5]
            
            print(f"  ✅ Feature İstatistikleri:")
            print(f"     En yüksek ortalama |SHAP|: {np.max(feature_means):.6f}")
            print(f"     En düşük ortalama |SHAP|: {np.min(feature_means):.6f}")
            print(f"     Feature'lar arası std oranı: {np.std(feature_means)/np.mean(feature_means):.3f}")
            
            # 4. Sample-wise Consistency
            sample_variances = np.var(class_shap, axis=(1, 2))  # Her sample için variance
            sample_means = np.mean(np.abs(class_shap), axis=(1, 2))
            
            print(f"  ✅ Sample İstatistikleri:")
            print(f"     Sample'lar arası SHAP variance ortalaması: {np.mean(sample_variances):.6f}")
            print(f"     Sample'lar arası |SHAP| ortalaması: {np.mean(sample_means):.6f}")
            print(f"     Sample consistency (CV): {np.std(sample_means)/np.mean(sample_means):.3f}")
            
            # 5. Temporal Pattern Analysis
            timestep_importance = np.mean(np.abs(class_shap), axis=(0, 2))  # Her timestep için ortalama
            
            # Gerçek uzunluklar içindeki pattern
            if hasattr(self, 'explain_real_lengths'):
                real_timestep_importance = []
                for t in range(n_timesteps):
                    # Bu timestep'te kaç sample'ın gerçek verisi var
                    valid_samples = [s for s in range(n_samples) if t < self.explain_real_lengths[s]]
                    if valid_samples:
                        valid_importance = np.mean([np.abs(class_shap[s, t, :]).mean() for s in valid_samples])
                        real_timestep_importance.append(valid_importance)
                    else:
                        real_timestep_importance.append(0)
                
                temporal_decay = np.corrcoef(range(len(real_timestep_importance)), real_timestep_importance)[0, 1]
                print(f"     Temporal importance korelasyonu: {temporal_decay:.3f}")
            
            # 6. Symmetry ve Linearity (approximations)
            # Random feature permutation ile symmetry test
            n_symmetry_tests = 10
            symmetry_violations = []
            
            for _ in range(n_symmetry_tests):
                # Random 2 feature seç
                f1, f2 = np.random.choice(n_features, 2, replace=False)
                
                # Bu feature'ların contribution'larının farkı
                contrib_diff = np.abs(class_shap[:, :, f1] - class_shap[:, :, f2])
                mean_contrib_diff = np.mean(contrib_diff)
                symmetry_violations.append(mean_contrib_diff)
            
            print(f"     Feature symmetry test ortalama fark: {np.mean(symmetry_violations):.6f}")
            
            # Statistics dictionary'e kaydet
            statistics[class_name] = {
                'efficiency_error_mean': np.mean(efficiency_errors),
                'efficiency_error_max': np.max(efficiency_errors),
                'temporal_violations': np.mean(temporal_violations) if temporal_violations else 0,
                'feature_importance_cv': np.std(feature_means)/np.mean(feature_means),
                'sample_consistency_cv': np.std(sample_means)/np.mean(sample_means),
                'top_features': [(self.feature_names[i] if self.feature_names else f'F_{i}', feature_means[i]) 
                               for i in top_features_idx[::-1]],
                'temporal_correlation': temporal_decay if hasattr(self, 'explain_real_lengths') else None,
                'symmetry_violation': np.mean(symmetry_violations)
            }
        
        # Genel özet
        print(f"\n🎯 GENEL SHAP DOĞRULAMA ÖZETİ:")
        
        overall_efficiency = np.mean([stats['efficiency_error_mean'] for stats in statistics.values()])
        overall_consistency = np.mean([stats['sample_consistency_cv'] for stats in statistics.values()])
        
        print(f"✅ Genel efficiency error: {overall_efficiency:.6f}")
        print(f"✅ Genel sample consistency: {overall_consistency:.3f}")
        
        # Doğrulama skorları
        efficiency_score = max(0, 1 - overall_efficiency * 1000)  # 0.001 error = 0 score
        consistency_score = max(0, 1 - overall_consistency)       # CV < 1 is good
        
        overall_score = (efficiency_score + consistency_score) / 2
        
        print(f"\n🏆 SHAP KALİTE SKORLARI:")
        print(f"   Efficiency Score: {efficiency_score:.3f}/1.0")
        print(f"   Consistency Score: {consistency_score:.3f}/1.0")
        print(f"   Overall Score: {overall_score:.3f}/1.0")
        
        if overall_score > 0.8:
            print("🎉 SHAP değerleri yüksek kalitede!")
        elif overall_score > 0.6:
            print("⚠️ SHAP değerleri kabul edilebilir kalitede.")
        else:
            print("❌ SHAP değerleri düşük kalitede, düzeltme gerekli!")
        
        return statistics, overall_score
    
    def plot_shap_summary_by_class(self, max_display=15, figsize=(12, 8), save_path=None):
        """Her sınıf için SHAP summary plot"""
        if not hasattr(self, 'shap_values'):
            raise ValueError("Önce run_shap_analysis() çağırın")
        
        if len(self.shap_values.shape) == 4:  # Multi-class
            n_classes = self.shap_values.shape[0]
            
            fig, axes = plt.subplots(n_classes, 1, figsize=(figsize[0], figsize[1] * n_classes))
            if n_classes == 1:
                axes = [axes]
            
            for i in range(n_classes):
                plt.sca(axes[i])
                
                # Zaman boyutunu ortalama alarak 2D'ye çevir
                shap_class_2d = np.mean(self.shap_values[i], axis=1)  # (n_samples, n_features)
                
                # Feature isimleri
                feature_names = self.feature_names if self.feature_names else [f'Feature_{j}' for j in range(shap_class_2d.shape[1])]
                
                shap.summary_plot(
                    shap_class_2d, 
                    features=self.X_explain.mean(axis=1),  # Zaman boyutunun ortalaması
                    feature_names=feature_names,
                    max_display=max_display,
                    show=False
                )
                plt.title(f'SHAP Summary - Class: {self.class_names[i]}')
        else:
            # Single output
            shap_2d = np.mean(self.shap_values, axis=1)
            feature_names = self.feature_names if self.feature_names else [f'Feature_{j}' for j in range(shap_2d.shape[1])]
            
            shap.summary_plot(
                shap_2d,
                features=self.X_explain.mean(axis=1),
                feature_names=feature_names,
                max_display=max_display,
                show=False
            )
            plt.title('SHAP Summary Plot')
        
        plt.tight_layout()
        
        final_save_path = self._get_save_path(save_path) if save_path else None
        if final_save_path:
            plt.savefig(final_save_path, dpi=300, bbox_inches='tight')
            print(f"Kaydedildi: {final_save_path}")
        plt.show()
    
    def plot_shap_violin_summary(self, class_idx=0, max_display=15, figsize=(12, 8), save_path=None):
        """
        SHAP Violin Plot - Feature değerlerinin dağılımını ve SHAP değerlerini birlikte gösterir
        
        Args:
            class_idx: Hangi sınıf için violin plot çizileceği
            max_display: Gösterilecek maksimum feature sayısı
        """
        if not hasattr(self, 'shap_values'):
            raise ValueError("Önce run_shap_analysis() çağırın")
        
        # SHAP değerlerini al
        if len(self.shap_values.shape) == 4:  # Multi-class
            shap_class = self.shap_values[class_idx]  # (n_samples, n_timesteps, n_features)
            class_name = self.class_names[class_idx]
        else:
            shap_class = self.shap_values  # (n_samples, n_timesteps, n_features)
            class_name = "Output"
        
        # Zaman boyutunu ortalama alarak 2D'ye çevir
        shap_2d = np.mean(shap_class, axis=1)  # (n_samples, n_features)
        features_2d = np.mean(self.X_explain, axis=1)  # (n_samples, n_features)
        
        # Feature isimleri
        feature_names = self.feature_names if self.feature_names else [f'Feature_{j}' for j in range(shap_2d.shape[1])]
        
        plt.figure(figsize=figsize)
        
        # SHAP violin plot
        shap.summary_plot(
            shap_2d,
            features=features_2d,
            feature_names=feature_names,
            max_display=max_display,
            plot_type="violin",  # Violin
            show=False
        )
        
        plt.title(f'SHAP Violin Plot - Class: {class_name}')
        plt.xlabel('SHAP value (impact on model output)')
        
        # Color bar label
        plt.gcf().axes[-1].set_ylabel('Feature value', size=12)
        
        plt.tight_layout()
        
        final_save_path = self._get_save_path(save_path) if save_path else None
        if final_save_path:
            plt.savefig(final_save_path, dpi=300, bbox_inches='tight')
            print(f"Kaydedildi: {final_save_path}")
        plt.show()
    
    def plot_shap_beeswarm(self, class_idx=0, max_display=15, figsize=(12, 8), save_path=None):
        """
        SHAP Beeswarm Plot - Her sample için feature katkılarını gösterir
        
        Args:
            class_idx: Hangi sınıf için beeswarm plot çizileceği
            max_display: Gösterilecek maksimum feature sayısı
        """
        if not hasattr(self, 'shap_values'):
            raise ValueError("Önce run_shap_analysis() çağırın")
        
        # SHAP değerlerini al
        if len(self.shap_values.shape) == 4:  # Multi-class
            shap_class = self.shap_values[class_idx]  # (n_samples, n_timesteps, n_features)
            class_name = self.class_names[class_idx]
        else:
            shap_class = self.shap_values  # (n_samples, n_timesteps, n_features)
            class_name = "Output"
        
        # Zaman boyutunu ortalama alarak 2D'ye çevir
        shap_2d = np.mean(shap_class, axis=1)  # (n_samples, n_features)
        features_2d = np.mean(self.X_explain, axis=1)  # (n_samples, n_features)
        
        # Feature isimleri
        feature_names = self.feature_names if self.feature_names else [f'Feature_{j}' for j in range(shap_2d.shape[1])]
        
        plt.figure(figsize=figsize)
        
        # SHAP beeswarm plot 
        try:
            shap.plots.beeswarm(
                shap.Explanation(
                    values=shap_2d,
                    data=features_2d,
                    feature_names=feature_names
                ),
                max_display=max_display,
                show=False
            )
        except:
            # Eski versiyon için fallback
            shap.summary_plot(
                shap_2d,
                features=features_2d,
                feature_names=feature_names,
                max_display=max_display,
                plot_type="dot",  # Dot plot
                show=False
            )
        
        plt.title(f'SHAP Beeswarm Plot - Class: {class_name}')
        plt.tight_layout()
        
        final_save_path = self._get_save_path(save_path) if save_path else None
        if final_save_path:
            plt.savefig(final_save_path, dpi=300, bbox_inches='tight')
            print(f"Kaydedildi: {final_save_path}")
        plt.show()

def corrected_comprehensive_analysis():
    """
    Kapsamlı SHAP analizi
    """
    model_dir = "trained_model_bilstm_attention_deneme_25_80296_27"
    
    # Analiz için ayarlar
    analyzer = TimeSeriesSHAPAnalyzer(
        model_dir, 
        model_name="bilstm_attention_model_epoch_27.keras",
        output_dir="shap_corrected_analysis_results"
    )
    
    analyzer.load_model_and_preprocessors()
    
    data_file = "balanced4000noresample_original_labels_increased_and_selected_and_filled_train_operational_readouts.csv"
    
    # Veri hazırlığı ile gerçek uzunlukları kaydet
    print("=== Veri Hazırlığı ===")
    analyzer.prepare_data_for_shap(
        data_file, 
        n_background=200,   ######
        n_explain=100       ######
    )
    
    # Masking SHAP analizi
    print("\n=== SHAP Analizi ===")
    
    try:
        # Masking yöntemi dene
        print("1️⃣ Feature-level masking...")
        analyzer._run_masking_shap(use_all_timesteps=True, use_feature_masking=True)
        analysis_type = "Corrected Feature-Level"
        
    except Exception as e:
        print(f"Feature-level masking hatası: {e}")
        try:
            print("2️⃣ Time-level masking...")
            analyzer._run_masking_shap(use_all_timesteps=True, use_feature_masking=False)
            analysis_type = "Corrected Time-Level"
            
        except Exception as e:
            print(f"Time-level masking hatası: {e}")
            print("3️⃣ Sampling fallback...")
            analyzer._run_sampling_shap()
            analysis_type = "Corrected Sampling"
    
    # DOĞRULAMA: İstatistiksel kontrol
    print("\n=== SHAP DOĞRULAMA ===")
    statistics, quality_score = analyzer.compute_corrected_shap_statistics()
    
#    if quality_score < 0.5:
#        print("⚠️ SHAP kalitesi düşük, alternatif yöntem deneniyor...")
#        try:
#            analyzer._run_sampling_shap()
#            statistics, quality_score = analyzer.compute_corrected_shap_statistics()
#            analysis_type += " + Sampling Fallback"
#        except:
#            print("❌ Alternatif yöntem de başarısız!")
    
    # Görselleştirmeler
    print(f"\n=== Görselleştirmeler ({analysis_type}) ===")
    
    # 1. Temporal analiz
    print("1️⃣ Temporal feature importance...")
    for i in range(min(3, analyzer.X_explain.shape[0])):
        for class_idx in range(min(2, len(analyzer.class_names))):
            analyzer.plot_feature_importance_over_time(
                sample_idx=i,
                class_idx=class_idx,
                top_features=15,
                save_path=f'corrected_temporal_sample_{i}_class_{class_idx}.png'
            )
    
    # 2. Sınıf analizleri
    print("2️⃣ Sınıf bazında analizler...")
    for i, class_name in enumerate(analyzer.class_names):
        analyzer.plot_shap_violin_summary(
            class_idx=i,
            max_display=20,
            save_path=f'corrected_violin_class_{i}_{class_name}.png'
        )
        
        analyzer.plot_shap_beeswarm(
            class_idx=i,
            max_display=20,
            save_path=f'corrected_beeswarm_class_{i}_{class_name}.png'
        )
    
    # 3. Genel özet
    analyzer.plot_shap_summary_by_class(
        max_display=25,
        save_path='corrected_summary_all_classes.png'
    )
    
    # 4. Kalite raporu oluştur
    print("\n=== KALİTE RAPORU ===")
    
    quality_report = f"""
SHAP ANALİZİ KALİTE RAPORU
=====================================

Analiz Tipi: {analysis_type}
Genel Kalite Skoru: {quality_score:.3f}/1.0

SINIF BAZINDA İSTATİSTİKLER:
"""
    
    for class_name, stats in statistics.items():
        quality_report += f"""
{class_name}:
  - Efficiency Error: {stats['efficiency_error_mean']:.6f} (max: {stats['efficiency_error_max']:.6f})
  - Temporal Violations: {stats['temporal_violations']:.6f}
  - Feature Consistency CV: {stats['feature_importance_cv']:.3f}
  - Sample Consistency CV: {stats['sample_consistency_cv']:.3f}
  - En Önemli Features: {', '.join([f[0][:15] for f in stats['top_features'][:3]])}
"""
    
    quality_report += f"""
ÖNERİLER:
"""
    
    if quality_score > 0.8:
        quality_report += "🎉 SHAP değerleri yüksek kalitede. Güvenle kullanılabilir.\n"
    elif quality_score > 0.6:
        quality_report += "⚠️ SHAP değerleri kabul edilebilir. Dikkatli yorumlama önerilir.\n"
    else:
        quality_report += "❌ SHAP değerleri düşük kalite. Daha fazla düzeltme gerekli.\n"
    
    if any(stats['efficiency_error_mean'] > 0.01 for stats in statistics.values()):
        quality_report += "• Efficiency error yüksek -> Daha fazla koalisyon sampling önerilir\n"
    
    if any(stats['temporal_violations'] > 0.001 for stats in statistics.values()):
        quality_report += "• Temporal violations var -> Padding detection iyileştirilebilir\n"
    
    if any(stats['sample_consistency_cv'] > 0.5 for stats in statistics.values()):
        quality_report += "• Sample tutarsızlığı yüksek -> Daha fazla background sample önerilir\n"
    
    # Raporu dosyaya kaydet
    report_path = os.path.join(analyzer.output_dir, 'quality_report.txt')
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(quality_report)
    
    print(quality_report)
    print(f"📄 Kalite raporu kaydedildi: {report_path}")
    
    # 5. Karşılaştırmalı analiz (eski vs yeni)
    print("\n=== KARŞILAŞTIRMALI ANALİZ ===")
    
    print("📊 Düzeltme öncesi vs sonrası beklenen farklar:")
    print("  • SHAP toplamları artık sıfıra daha yakın olmalı")
    print("  • Padding bölgelerinde SHAP değerleri sıfıra yakın olmalı")
    print("  • Feature importance'lar temporal pattern'ları korumalı")
    print("  • Background sampling daha reprezentatif olmalı")
    print("  • Koalisyon tabanlı masking daha doğru attribution vermeli")
    
    # 6. İleri analiz önerileri
    print("\n=== İLERİ ANALİZ ÖNERİLERİ ===")
    
    if quality_score > 0.7:
        print("🚀 Kalite yüksek olduğu için ileri analizler yapılabilir:")
        print("  • Attention weight'leri ile SHAP değerlerini karşılaştırın")
        print("  • Feature interaction analizi yapın")
        print("  • Temporal dependency patterns araştırın")
        print("  • Cost-sensitive SHAP analizi geliştirin")
    
    # 7. Performans metrikleri
    print(f"\n=== PERFORMANS METRİKLERİ ===")
    print(f"✅ Analiz tamamlandı!")
    print(f"Background samples: {analyzer.X_background.shape[0]}")
    print(f"Explain samples: {analyzer.X_explain.shape[0]}")
    print(f"Ortalama gerçek sequence uzunluğu: {np.mean(analyzer.explain_real_lengths):.1f}")
    print(f"SHAP shape: {analyzer.shap_values.shape}")
    print(f"Kalite skoru: {quality_score:.3f}/1.0")
    print(f"Sonuçlar: '{analyzer.output_dir}' klasöründe")
    
    # 8. Dosya listesi
    print(f"\n📁 OLUŞTURULAN DOSYALAR:")
    output_files = [
        "quality_report.txt",
        "corrected_summary_all_classes.png",
    ]
    
    # Temporal analiz dosyaları
    for i in range(min(3, analyzer.X_explain.shape[0])):
        for class_idx in range(min(2, len(analyzer.class_names))):
            output_files.append(f"corrected_temporal_sample_{i}_class_{class_idx}.png")
    
    # Sınıf analiz dosyaları
    for i, class_name in enumerate(analyzer.class_names):
        output_files.append(f"corrected_violin_class_{i}_{class_name}.png")
        output_files.append(f"corrected_beeswarm_class_{i}_{class_name}.png")
    
    for file in output_files[:10]:  # İlk 10 dosyayı listele
        print(f"  ├── {file}")
    
    if len(output_files) > 10:
        print(f"  └── ... ve {len(output_files)-10} dosya daha")
    
    print(f"\n🎉 SHAP ANALİZİ TAMAMLANDI!")
    print(f"🔍 Kalite skoru: {quality_score:.3f}/1.0")
    print(f"📈 Kullanılan yöntem: {analysis_type}")
    
    return analyzer, statistics, quality_score

def corrected_main():
    """Ana fonksiyon"""
    try:
        analyzer, statistics, quality_score = corrected_comprehensive_analysis()
        
        print("\n" + "="*50)
        print("DÜZELTME ÖZETİ BAŞARILI!")
        print("="*50)
        print(f"Final Kalite Skoru: {quality_score:.3f}/1.0")
        
        if quality_score > 0.8:
            print("🎉 Mükemmel! SHAP analizi artık güvenilir.")
        elif quality_score > 0.6:
            print("✅ İyi! SHAP analizi kabul edilebilir kalitede.")
        else:
            print("⚠️ Dikkat! Daha fazla iyileştirme gerekebilir.")
            
    except Exception as e:
        print(f"❌ Analiz hatası: {e}")
        print("🔄 Lütfen parametreleri kontrol edin ve tekrar deneyin.")

corrected_main()
